# 🛠️ Diabetes Risk Predictor: a DevOps demo

The MLOps demo (`demo/mlops_demo.ipynb`) showed how the **model** is built, served and kept up to date.
This notebook shows how the **software around it** is changed safely: every change is checked automatically,
shipped automatically, and watched once it's live. That way of working is called **DevOps**
(development + operations): small changes, automated checks, and fast, safe releases.

About 8 minutes. Every step runs the project's **real** tools; where something lives on GitHub or in the cloud,
a 🎤 cue says when to switch browser tabs.

**How to read this notebook**
- **Talking points** cells explain each idea in plain language (they double as a handout).
- Lines starting with 🎤 are cues for the presenter.
- If a live step fails, a clearly labelled result saved during a rehearsal is shown instead.

## 0. Before the audience arrives *(presenter only, not presented)*

🎤 **Checklist**, about 5 minutes before you start:

1. **Docker Desktop** running, then in a terminal: `docker compose up --build` (the API and web app)
2. **Browser tabs:**
   - your GitHub repository: a **prepared pull request** (any small change) showing its CI checks
   - the repository's **Actions** tab, with a finished CI run open on its **Summary** page
   - a **Dependabot** pull request, if one exists
   - the **Render** dashboard for the API (Events tab shows deploys; the rollback button lives there)
   - the deployed web app
3. Fill in the settings below, then run every cell in this section. All checks should show ✅.

**First time?** Rehearse once with `MODE = "record"` to save the fallback results.

In [ ]:
# Make the project folder the working directory, so the project's code can be imported.
# (VS Code may start notebooks inside the demo/ folder.)
import os
import sys

if os.path.basename(os.getcwd()) == "demo":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
from demo import common
from demo import devops_helpers as devops

# ---- Settings ---------------------------------------------------------------
# "live"   run for real; if a step fails, show the saved rehearsal result
# "record" rehearsal: run for real AND save results as the fallback
# "saved"  run nothing live; show saved results only (safest)
MODE = "live"

API_URL = "http://localhost:8000"  # local API container (docker compose)
UI_URL = "http://localhost:8501"  # local web app container
DEPLOYED_API_URL = "https://<your-service>.onrender.com"  # your Render URL
DEPLOYED_UI_URL = "https://<your-app>.streamlit.app"  # your Streamlit Cloud URL

common.configure()  # quiet, predictable output
common.wake_deployed_api(DEPLOYED_API_URL)  # free tier sleeps: start waking it now
runner = common.DemoRunner(MODE, devops.FALLBACK_DIR)  # saved results live in demo/fallback/devops

In [ ]:
# Is everything ready? Each ❌ shows how to fix it.
devops.preflight(API_URL, UI_URL, DEPLOYED_API_URL, DEPLOYED_UI_URL)

## 1. The big picture

**Talking points**
- Software changes all the time: fixes, features, updated libraries, new models. Each change could break something.
- DevOps answers: **change in small steps, check every step automatically, and make releasing boring.**
- Here's the path every change in this project takes:

```
  💻 Code ─▶ 🧹 Pre-commit checks ─▶ 🔀 Pull request ─▶ 🤖 CI checks ─▶ ✅ Merge
                                                                         │
  🔄 Update ◀─ 📈 Monitor ◀─ 🔍 Verify (smoke test) ◀─ 🚀 Automatic deploy ◀┘
```

We'll follow one change along that path.

🎤 ⏱ 1 min · no code to run here

## 2. Everything as code, one home per setting

**Talking points**
- Every part of this system is described in **files in Git**: the code, the packages it needs, how it's tested,
  how it's packaged, even the cloud server. Nothing depends on someone remembering which buttons they clicked.
  Describing servers this way is called **infrastructure as code**.
- Every value has **exactly one home**. Change it there, and every part of the system follows.
  A number typed in two places will eventually disagree with itself.

The table is read **live** from the real files right now:

🎤 ⏱ 1 min

In [ ]:
# Read example settings live from their home files
common.show_table(
    devops.settings_homes(),
    {
        "setting": "Setting",
        "value": "Current value",
        "lives_in": "Its one home",
        "kind": "Kind of setting",
    },
)

## 3. Catching problems before they're committed

**Talking points**
- The cheapest bug to fix is the one caught seconds after it was typed.
- Before every commit, **pre-commit hooks** run automatically on the changed files:
  - a **formatter** rewrites code into one consistent layout (no style arguments),
  - a **linter** reads code and reports likely bugs, like an import that's never used.
- Here's a deliberately messy file, and what those tools say about it:

🎤 ⏱ 1 min

In [ ]:
# Run the real formatter and linter on a messy file (in a temporary folder)
lint = runner.run("lint", devops.lint_demo)
print("📄 The messy file:\n")
print(lint["before"])
print("🔎 The linter's findings:\n")
print(lint["lint_findings"])

In [ ]:
# The formatter's proposed changes: lines starting with "-" go, lines with "+" replace them
print(lint["format_diff"])

## 4. Continuous integration (CI) and test coverage

**Talking points**
- Changes go into the main code only through a **pull request**: a proposal that says "please merge my change".
- Every pull request triggers **CI**: GitHub starts fresh computers and runs five checks, listed below
  (read live from the workflow file `.github/workflows/ci.yml`).
- **Branch protection** means `main` refuses any change whose required checks didn't pass. No exceptions,
  not even for the project's owner.
- One of the checks measures **test coverage**: which share of the code actually ran during the tests.
  Code no test runs could be broken without anyone noticing.

🎤 ⏱ 1.5 min · the coverage run takes ~20 seconds

In [ ]:
# The CI pipeline, read live from .github/workflows/ci.yml
common.show_table(devops.ci_jobs(), {"job": "Job", "steps": "What it does, step by step"})

In [ ]:
# Run the test suite with coverage, exactly as CI does
cov = runner.run("coverage", devops.coverage_summary)
status = "✅ above" if cov["total"] >= cov["minimum"] else "❌ below"
print(f"Test coverage: {cov['total']}% of the code ran during the tests ({status} the {cov['minimum']}% minimum)")
print("\nLeast-covered files (where new tests would help most):")
for row in cov["lowest"]:
    print(f"  {row['covered']:>3}%  {row['file']}")

**About that number:** coverage shows which code **ran** during the tests, not whether the tests **checked** the
right things. So it's a guide for finding untested code, not a goal in itself. The minimum is 80%, not 100%:
chasing the last few percent tends to produce meaningless tests.

🎤 Switch to the **prepared pull request** on GitHub: show the five checks (✅/❌), then the branch-protection
message ("Required checks must pass before merging"). Open a finished run's **Summary** page: the coverage table and
the security-scan results are there.

## 5. The model quality gate

**Talking points**
- In an ML system, the **model** is part of the release too. So CI also checks the model, not just the code.
- The **quality gate** is a minimum standard a model must meet before it may serve anyone: it must load with the
  right library version, rank people clearly better than a coin flip, beat simply guessing the average, and give
  sensible probabilities.
- The same gate runs in two places: before the pipeline **promotes** a model, and in CI before the app is
  **deployed**. One rule, applied everywhere.
- First, the real production model. Then a deliberately useless "model" that gives everyone the same answer:

🎤 ⏱ 1.5 min

In [ ]:
# The real production model, through the real gate
gate = runner.run("gate_production", devops.gate_on_production_model)
devops.print_checks_report(gate, "Quality gate: the production model")

In [ ]:
# A useless model that ignores every answer and predicts the average for everyone
useless = runner.run("gate_useless", devops.gate_on_useless_model)
devops.print_checks_report(useless, "Quality gate: a useless model")

## 6. Deploy and verify

**Talking points**
- When a change is merged, **deployment is automatic**: Render notices the new commit, waits for CI to pass
  (configured in `render.yaml`), builds the Docker image and swaps it in, only once the new version reports healthy.
- "It deployed" isn't the same as "it works". A **smoke test** checks the running service from the outside, like a
  user would: is it alive, does it answer sensibly, does it refuse nonsense?
- CI runs this exact smoke test against freshly built containers. Here it is against the local containers,
  then against the real deployed app:

🎤 ⏱ 1 min

In [ ]:
# The real smoke test (scripts/smoke_test.py) against the local containers
local = runner.run("smoke_local", lambda: devops.smoke(API_URL, UI_URL, deployed=False))
devops.print_checks_report(local, "Smoke test: local containers")

In [ ]:
# The same smoke test against the live, deployed app
live = runner.run("smoke_deployed", lambda: devops.smoke(DEPLOYED_API_URL, DEPLOYED_UI_URL, deployed=True))
devops.print_checks_report(live, "Smoke test: deployed app")
print(f"\nRunning in production: app v{live['app_version']}, model v{live['model_version']}")

🎤 Switch to the **Render** dashboard: show the deploy events (each one waited for CI) and the **rollback** button.

## 7. Keeping it healthy

**Talking points**
- **Dependency updates:** a robot (Dependabot) opens pull requests every week when newer versions of our packages
  exist. CI tests each one like any other change, so updating is small and routine instead of big and scary.
- **Security scans:** CI checks our packages and our Docker image against public lists of known vulnerabilities.
  This project reports findings as warnings on the run's Summary page; fixes arrive through Dependabot.
- **Production check:** every morning, GitHub runs the smoke test against the live app: a free uptime check.
  If it fails, the owner gets an email.
- **Rollback:** if a release goes wrong, undo it. For code: Render's rollback button or `git revert`.
  For models: because the model is committed to Git, every model release is a commit, so reverting one is easy:

🎤 ⏱ 1 min · optional: show a Dependabot pull request and a Production check run on the Actions tab

In [ ]:
# Every model release is a Git commit: this is the model's release history
history = runner.run("model_history", lambda: {"commits": devops.model_history()})
for line in history["commits"]:
    print(line)

## 8. Wrap-up

**Talking points**
- One change's journey: checked on the laptop (pre-commit) → checked by CI (format, tests, coverage, model gate,
  security, containers) → merged only if it passed (branch protection) → deployed automatically → verified by a
  smoke test → watched every day → easy to undo.
- None of this needs a person to remember anything. That's the point: **the process is the safety net.**
- Together with the MLOps demo, that's the whole system: a model that stays good, in software that stays healthy.

🎤 ⏱ 30 sec

In [ ]:
# Presenter: before committing this notebook as a handout, every step should have run live.
runner.handout_check()

## Glossary

| Term | Meaning |
|---|---|
| **DevOps** | Development + operations: small changes, automated checks, fast and safe releases |
| **Git / commit** | Version control: every change is saved as a commit, with its history |
| **Branch** | A separate line of work; changes are made on a branch, then merged into `main` |
| **Pull request** | A proposal to merge a branch into `main`, where it's checked and reviewed |
| **Branch protection** | A rule that `main` only accepts changes whose required checks passed |
| **Pre-commit hook** | A check that runs automatically on your machine before each commit |
| **Formatter / linter** | Tools that tidy code layout / find likely bugs by reading code |
| **CI (continuous integration)** | Automatically checking every change (tests, lint, builds) |
| **CD (continuous deployment)** | Automatically releasing every change that passed CI |
| **Workflow / job / runner** | A CI recipe / one group of steps in it / the temporary computer that runs it |
| **Test coverage** | The share of code that ran during the tests; a guide to untested code, not a goal |
| **Quality gate** | A minimum standard a model must meet before it may be promoted or deployed |
| **Container / image** | A packaged app with everything it needs / the package a container is started from |
| **Smoke test** | A quick check that a running service is alive and answers sensibly |
| **Infrastructure as code** | Describing servers and services in files kept in Git (here: `render.yaml`) |
| **Lockfile** | A file recording the exact version of every package (`uv.lock`), for identical installs |
| **Environment parity** | Laptop, CI and production use the same versions, so "works here" means "works there" |
| **Vulnerability scan** | Comparing package versions against public lists of known security flaws |
| **Dependabot** | A GitHub robot that opens pull requests to update dependencies |
| **Semantic versioning** | MAJOR.MINOR.PATCH version numbers: breaking change . new feature . fix |
| **Rollback** | Undoing a release by returning to the previous working version |